# =========================
# INSTALLS
# =========================

In [ ]:

!pip install --no-index /kaggle/input/datasets/kurshidbasheer/biopython-offline/biopython-1.83-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!pip install --no-index /kaggle/input/datasets/kurshidbasheer/pyg-2-7-torch-2-9-cpu-py312-kur/torch_geometric-2.7.0-py3-none-any.whl


# =========================
# IMPORTS
# =========================

In [ ]:
import torch, random
import numpy as np
import pandas as pd
import torch.nn as nn
from collections import defaultdict

from tqdm.auto import tqdm

from torch.utils.data import Dataset
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import scatter

# =========================
# DEVICE
# =========================

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================
# PATHS
# =========================

In [ ]:
TRAIN_SEQ = "/kaggle/input/competitions/stanford-rna-3d-folding-2/train_sequences.csv"
TRAIN_LBL = "/kaggle/input/competitions/stanford-rna-3d-folding-2/train_labels.csv"
TEST_SEQ  = "/kaggle/input/competitions/stanford-rna-3d-folding-2/test_sequences.csv"

# =========================
# GRAPH AND WINDOW SIZING
# =========================

In [ ]:
WINDOW_SIZE = 400
STRIDE = 200
K_NEIGHBORS = 16

# =========================
# UTILS
# =========================

In [ ]:
NUC_MAP = {'A':0,'U':1,'G':2,'C':3}

def clean_sequence(seq):
    return "".join([s for s in seq.upper() if s in NUC_MAP])

def one_hot(seq):
    x = torch.zeros(len(seq),4)
    for i,s in enumerate(seq):
        x[i,NUC_MAP[s]] = 1
    return x

def center_coords(x):
    return x - x.mean(0, keepdim=True)

# =========================
# GRAPH
# =========================

In [ ]:
def build_graph(x, coords=None, k=16):
    L = x.size(0)
    row, col = [], []

    for i in range(L):
        local = list(range(max(0,i-k), min(L,i+k+1)))

        # 🔥 reduced randomness (important fix)
        rand_nodes = random.sample(range(L), min(k//2, L))

        neigh = list(set(local + rand_nodes))
        if i in neigh: neigh.remove(i)

        row += [i]*len(neigh)
        col += neigh

    edge_index = torch.tensor([row,col], dtype=torch.long)

    rel = (edge_index[0]-edge_index[1]).float().unsqueeze(1)/L
    dist = rel.abs()

    edge_attr = torch.cat([dist, rel], dim=1)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

    if coords is not None:
        coords = center_coords(coords)  # 🔥 normalize coords
        data.y = coords
        data.pos = coords + 0.01*torch.randn_like(coords)
    else:
        data.pos = torch.randn(L,3)*0.1

    return data

# =========================
# DATASET
# =========================

In [ ]:
class RNAWindowDataset(Dataset):
    def __init__(self, seq_csv, label_csv=None):
        self.df = pd.read_csv(seq_csv)
        self.has_labels = label_csv is not None

        self.seq_map = {
            row["target_id"]: clean_sequence(row["sequence"])
            for _,row in self.df.iterrows()
        }

        if self.has_labels:
            labels = pd.read_csv(label_csv, low_memory=False)  # 🔥 fix warning

            labels["sid"] = labels["ID"].str.split("_").str[0]
            labels["idx"] = labels["ID"].str.split("_").str[1].astype(int)

            self.coords = {}
            for k,g in labels.groupby("sid"):
                g = g.sort_values("idx")
                xyz = torch.tensor(g[["x_1","y_1","z_1"]].values, dtype=torch.float32)

                valid = ~torch.isnan(xyz).any(dim=1)
                xyz = xyz[valid]

                if len(xyz)>0:
                    xyz = center_coords(xyz)  # 🔥 important
                    self.coords[k] = xyz

        self.samples = []

        for sid in self.df["target_id"]:
            seq = self.seq_map[sid]
            L = len(seq)

            if self.has_labels and sid in self.coords:
                L = min(L, self.coords[sid].shape[0])

            for s in range(0, L, STRIDE):
                e = min(s+WINDOW_SIZE, L)
                if e-s >= 20:
                    self.samples.append((sid,s,e))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sid,s,e = self.samples[idx]
        seq = self.seq_map[sid][s:e]

        coords = None
        if self.has_labels and sid in self.coords:
            coords = self.coords[sid][s:e]

        x = one_hot(seq)
        pos_feat = torch.arange(len(seq)).float().unsqueeze(-1)/len(seq)
        x = torch.cat([x,pos_feat],dim=1)

        return build_graph(x, coords)

# =========================
# KABSCH
# =========================

In [ ]:
def kabsch(P, Q):
    Pc = P - P.mean(0,keepdim=True)
    Qc = Q - Q.mean(0,keepdim=True)

    C = Pc.t() @ Qc
    U,S,Vt = torch.linalg.svd(C)

    d = torch.det(U@Vt)
    D = torch.eye(3, device=P.device)
    D[-1,-1] = d

    R = U @ D @ Vt
    return Pc @ R


# =========================
# TM LOSS
# =========================

In [ ]:
def tm_loss(pred, target, batch):
    loss = 0
    n = batch.max()+1

    for i in range(n):
        mask = batch==i
        P = pred[mask]
        Q = target[mask]

        with torch.amp.autocast("cuda", enabled=False):
            P = kabsch(P, Q)

        L = P.shape[0]

        if L >= 30:
            d0 = 1.24*(L-15)**(1/3) - 1.8
        else:
            d0 = 0.5

        d = torch.norm(P-Q, dim=1)
        tm = (1/(1+(d/d0)**2)).mean()

        loss += (1 - tm)

    return loss/n

# =========================
# MODEL
# =========================

In [ ]:
class EGNNLayer(nn.Module):
    def __init__(self, hidden):
        super().__init__()

        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden*2+2, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden)
        )

        self.node_mlp = nn.Sequential(
            nn.Linear(hidden*2, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden)
        )

        self.coord_mlp = nn.Sequential(
            nn.Linear(hidden,1),
            nn.Tanh()
        )

    def forward(self,x,pos,edge_index,edge_attr):
        row,col = edge_index
        rel = pos[row]-pos[col]

        m = self.edge_mlp(torch.cat([x[row],x[col],edge_attr],dim=1))
        agg = scatter(m,row,dim=0,dim_size=x.size(0),reduce="mean")

        x = self.node_mlp(torch.cat([x,agg],dim=1))

        trans = self.coord_mlp(m)*rel
        delta = scatter(trans,row,dim=0,dim_size=pos.size(0),reduce="mean")

        pos = pos + delta
        return x,pos

class EGNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Linear(5,128)
        self.layers = nn.ModuleList([EGNNLayer(128) for _ in range(6)])
        self.dropout = nn.Dropout(0.1)

    def forward(self,data):
        x,pos = data.x,data.pos
        x = self.emb(x)

        for l in self.layers:
            x,pos = l(x,pos,data.edge_index,data.edge_attr)
            x = self.dropout(x)

        return pos

# =========================
# TRAIN
# =========================

In [ ]:
def train_epoch(model, loader, opt):
    model.train()
    total = 0

    pbar = tqdm(loader, leave=False)

    for data in pbar:
        data = data.to(DEVICE)
        opt.zero_grad()

        pred = model(data)

        # 🔥 hybrid loss (CRITICAL FIX)
        loss_tm = tm_loss(pred, data.y, data.batch)
        loss_mse = ((pred - data.y)**2).mean()

        loss = loss_tm + 0.3 * loss_mse

        loss.backward()
        opt.step()

        total += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total / len(loader)

# =========================
# INFERENCE
# =========================

In [ ]:
def run_inference(model, dataset):
    storage = defaultdict(list)

    for sid,s,e in dataset.samples:
        seq = dataset.seq_map[sid][s:e]

        x = one_hot(seq)
        pos_feat = torch.arange(len(seq)).float().unsqueeze(-1)/len(seq)
        x = torch.cat([x,pos_feat],dim=1)

        g = build_graph(x, None)
        data = g.to(DEVICE)

        preds = []

        for _ in range(5):
            model.train()
            with torch.no_grad():
                preds.append(model(data).cpu())

        storage[sid].append((s,e,preds))

    return storage

# =========================
# MERGE
# =========================

In [ ]:
def merge_windows(storage):
    final={}

    for sid,chunks in storage.items():
        L = max(e for _,e,_ in chunks)

        coords = [torch.zeros(L,3) for _ in range(5)]
        counts = torch.zeros(L,1)

        for s,e,preds in chunks:
            w = torch.linspace(0.5,1.0,e-s).unsqueeze(1)

            for k in range(5):
                coords[k][s:e] += preds[k]*w

            counts[s:e]+=w

        final[sid] = [c/counts.clamp(min=1) for c in coords]

    return final

# =========================
# SUBMISSION
# =========================

In [ ]:
def build_submission(test_ds, preds):
    rows=[]

    for sid in test_ds.df["target_id"]:
        seq = test_ds.seq_map[sid]
        coords_list = preds[sid]

        for i in range(len(seq)):
            r={"ID":f"{sid}_{i+1}","resname":seq[i],"resid":i+1}

            for k in range(5):
                c = coords_list[k]
                r[f"x_{k+1}"]=float(c[i,0])
                r[f"y_{k+1}"]=float(c[i,1])
                r[f"z_{k+1}"]=float(c[i,2])

            rows.append(r)

    pd.DataFrame(rows).to_csv("submission.csv",index=False)

# =========================
# RUN
# =========================

In [ ]:
train_ds = RNAWindowDataset(TRAIN_SEQ, TRAIN_LBL)
test_ds  = RNAWindowDataset(TEST_SEQ)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)

model = EGNNModel().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=3e-4)

for e in tqdm(range(10), desc="Training"):
    loss = train_epoch(model, train_loader, opt)
    print(f"Epoch {e} | loss = {loss:.6f}")

print("Inference...")
storage = run_inference(model, test_ds)

print("Merging...")
merged = merge_windows(storage)

print("Submission...")
build_submission(test_ds, merged)

print("DONE ✅")